# Study 924 — First Cut — the teardown

Event study on a hardcoded FOMC cycle-start calendar: buy TLT at the close of *t+1* after the announcement (one execution lag, no exceptions), hold 1/3/6/12 months, score excess of BIL over the identical days, 5 bps one-way each leg. Below: the per-event table, the horizon sweep, the all-cuts control, the randomisation placebo, the daily conditional leg with its HAC *t* and block-bootstrap CI, the era split, the cost and borrow sweeps, the IEF and `^IRX` cross-checks, and a power calibration of the five-event test itself.

Every real number is frozen from `docs/results.md` (Fingerprint `ff993b355f57`), as-of 2026-06-30.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'fp': 'ff993b355f57', 'n_first': 5, 'n_all': 31, 'n_events': 4, 'events': [('2007-09-18', '2007-09-19', '2008-09-18', 14.37, 2.78, 11.5), ('2019-07-31', '2019-08-01', '2020-07-31', 28.58, 1.11, 27.37), ('2020-03-03', '2020-03-04', '2021-03-03', -8.51, 0.05, -8.67), ('2024-09-18', '2024-09-19', '2025-09-18', -6.2, 4.36, -10.66)], 'ex_2019_mean': -2.61, 'h_first': {1: (3.18, 3.08, 4.61, 0.91, 0.75, 40.13, 2.22, 1.04), 3: (0.42, 0.32, 3.71, 0.12, 0.75, 3.44, 2.86, 0.18), 6: (3.79, 3.69, 7.99, 0.9, 0.75, 8.67, 2.22, 0.81), 12: (4.98, 4.88, 1.51, 0.55, 0.5, 2.62, 2.95, 0.35)}, 'h_all': {1: (0.96, 0.86, 0.8, 0.44, 0.78), 3: (0.54, 0.44, 0.31, 0.5, 0.6), 6: (1.3, 1.2, 0.64, 0.61, 0.65), 12: (5.6, 5.5, 1.78, 0.72, 0.22)}, 'n_all_events': 18, 'placebo': {1: (3.08, 0.14, 2.04, 0.086), 3: (0.32, 0.72, 3.54, 0.54), 6: (3.69, 1.34, 5.14, 0.319), 12: (4.88, 2.61, 7.18, 0.365)}, 'cond_ann': 0.47, 'cond_lo': -2.05, 'cond_hi': 3.02, 'cond_negfrac': 36.6, 'in_frac': 18.7, 'in_ann_pct12': 2.62, 'uncond_ann': 2.89, 'uncond_lo': -3.64, 'uncond_hi': 9.04, 'n_episodes_first': 3, 'n_episodes_all': 3, 'overlap_pairs_all': 53, 'overlap_pairs_all_tot': 153, 'era_early': 19.43, 'era_late': -9.66, 'cost0': 4.98, 'cost5': 4.88, 'cost10': 4.78, 'cost25': 4.48, 'cost25_t': 0.5, 'ls_b0': 3.01, 'ls_b0_t': 0.36, 'ls_b50': 2.51, 'ls_b50_t': 0.3, 'ls_b200': 1.01, 'ls_b200_t': 0.12, 'ief_12m': 4.26, 'ief_12m_t': 1.17, 'ief_6m': 2.98, 'ief_6m_t': 1.06, 'irx_mean': 5.09, 'irx_t': 0.56, 'syn_planted_true': 9.0, 'syn_planted_pooled': 8.55, 'syn_planted_t': 6.93, 'syn_null_pooled': -0.83, 'syn_null_t': -0.74, 'syn_planted_fire': 5, 'syn_null_fire': 1, 'syn_worlds': 12}

## Design and its one non-tape input

- **Execution lag:** signal formed at the close of the announcement day *t*, position opened at the **close of t+1**. Exactly one lag, applied identically to the real, control, placebo and synthetic paths.
- **Costs:** 5 bps one-way × NAV on entry and on exit (two legs; four for the curve expression). Swept.
- **Excess of cash:** BIL's own total return over the same days, not a flat proxy — at 2024-26 bill yields that distinction is worth ~4.4 points on a 12-month window.
- **Total return throughout**, never price-only: coupon is most of a bond ETF's 12-month return.
- **ASSUMPTION (the only one):** the event calendar is hand-typed — 5 cycle-start cuts, 31 cuts in total, truncated at 2024-12-18 so every listed event has a full 12-month window inside the as-of.
- **Survivorship:** none — single still-listed ETFs, calendar fixed ex ante. The selection risk is of another kind: *which* cuts count as "first" is a judgement, which is why the all-cuts control exists.

## Per-event table — the whole sample, 12-month hold

In [2]:
events = [('2007-09-18', '2007-09-19', '2008-09-18', 14.37, 2.78, 11.5), ('2019-07-31', '2019-08-01', '2020-07-31', 28.58, 1.11, 27.37), ('2020-03-03', '2020-03-04', '2021-03-03', -8.51, 0.05, -8.67), ('2024-09-18', '2024-09-19', '2025-09-18', -6.2, 4.36, -10.66)]
print(f"{'first cut':12s} {'entry':12s} {'exit':12s} {'TLT':>8s} {'BIL':>7s} {'excess net':>11s}")
for ev, entry, ex, tlt, bil, net in events:
    print(f'{ev:12s} {entry:12s} {ex:12s} {tlt:+8.2f} {bil:+7.2f} {net:+11.2f}')
mean = sum(e[5] for e in events) / len(events)
print(f'\nN = {len(events)}   mean excess net = {mean:+.2f}%   hit rate = {sum(1 for e in events if e[5] > 0)}/{len(events)}')
print('2001-01-03 is unmeasurable: TLT only lists from 2002-07-30.')

first cut    entry        exit              TLT     BIL  excess net
2007-09-18   2007-09-19   2008-09-18     +14.37   +2.78      +11.50
2019-07-31   2019-08-01   2020-07-31     +28.58   +1.11      +27.37
2020-03-03   2020-03-04   2021-03-03      -8.51   +0.05       -8.67
2024-09-18   2024-09-19   2025-09-18      -6.20   +4.36      -10.66

N = 4   mean excess net = +4.88%   hit rate = 2/4
2001-01-03 is unmeasurable: TLT only lists from 2002-07-30.


> 💡 **In plain words** — four trades. Two won, two lost, and the average is carried by a window that happens to contain March 2020.

## Horizon sweep: first cuts vs the all-cuts control

`in`/`out` are annualised *conditional* rates on the daily excess-of-cash series (post-cut days vs every other day) — not the return of any fund. The HAC *t* is Newey-West on the daily conditional leg.

In [3]:
print(f"{'h':>3s} {'FIRST net':>10s} {'t(N=4)':>7s} {'hit':>5s} {'in %/y':>8s} "
      f"{'out %/y':>8s} {'HAC t':>6s} | {'ALL net':>8s} {'t(N=18)':>8s} {'hit':>5s}")
for h in (1, 3, 6, 12):
    f = R['h_first'][h]; a = R['h_all'][h]
    print(f'{h:>2d}m {f[1]:>+10.2f} {f[3]:>+7.2f} {f[4]:>5.0%} {f[5]:>+8.2f} '
          f'{f[6]:>+8.2f} {f[7]:>+6.2f} | {a[1]:>+8.2f} {a[2]:>+8.2f} {a[3]:>5.0%}')
print('\nAt 12m the ALL-cuts leg (+5.50%, t=+1.78, N=18) beats the FIRST-cut leg '
      '(+4.88%, t=+0.55, N=4).')

  h  FIRST net  t(N=4)   hit   in %/y  out %/y  HAC t |  ALL net  t(N=18)   hit
 1m      +3.08   +0.91   75%   +40.13    +2.22  +1.04 |    +0.86    +0.80   44%
 3m      +0.32   +0.12   75%    +3.44    +2.86  +0.18 |    +0.44    +0.31   50%
 6m      +3.69   +0.90   75%    +8.67    +2.22  +0.81 |    +1.20    +0.64   61%
12m      +4.88   +0.55   50%    +2.62    +2.95  +0.35 |    +5.50    +1.78   72%

At 12m the ALL-cuts leg (+5.50%, t=+1.78, N=18) beats the FIRST-cut leg (+4.88%, t=+0.55, N=4).


> 💡 **In plain words** — if the special ingredient were the *first* cut, the narrow list should beat the broad one. It does not, at any horizon worth trading.

## Randomisation inference — the only test N=4 can honestly support

2,000 draws of N random eligible start dates, same horizon, same costs. The *p* is the share of draws whose mean is at least as good as the observed one.

In [4]:
print(f"{'h':>3s} {'observed':>9s} {'placebo mu':>11s} {'placebo sd':>11s} {'one-sided p':>12s}")
for h in (1, 3, 6, 12):
    o, mu, sd, p = R['placebo'][h]
    print(f'{h:>2d}m {o:>+9.2f} {mu:>+11.2f} {sd:>11.2f} {p:>12.3f}')
print('\nFour horizons examined; the best p is 0.086 at 1m and survives no '
      'multiplicity adjustment at all.')
print('Caveat: the draws are iid dates, while the real events cluster in three '
      'episodes. A clustered null would be wider, so these p-values are a '
      'FLOOR - the true p is larger, which only reinforces the None stamp.')

  h  observed  placebo mu  placebo sd  one-sided p
 1m     +3.08       +0.14        2.04        0.086
 3m     +0.32       +0.72        3.54        0.540
 6m     +3.69       +1.34        5.14        0.319
12m     +4.88       +2.61        7.18        0.365

Four horizons examined; the best p is 0.086 at 1m and survives no multiplicity adjustment at all.
Caveat: the draws are iid dates, while the real events cluster in three episodes. A clustered null would be wider, so these p-values are a FLOOR - the true p is larger, which only reinforces the None stamp.


## The daily conditional leg — HAC *t* and block-bootstrap CI

Per-event returns from overlapping windows are not independent, so the inferential spine is the daily excess-of-cash series gated by the post-cut window: Newey-West *t* on the mean, circular block bootstrap (2,000 draws, 21-day blocks) on the CI.

In [5]:
print(f"conditional leg (12m windows): {R['cond_ann']:+.2f}%/y  "
      f"95% CI [{R['cond_lo']:+.2f}, {R['cond_hi']:+.2f}]  share<0 {R['cond_negfrac']:.1f}%  "
      f"invested {R['in_frac']:.1f}% of days")
print(f"unconditional TLT - BIL      : {R['uncond_ann']:+.2f}%/y  "
      f"95% CI [{R['uncond_lo']:+.2f}, {R['uncond_hi']:+.2f}]")
print(f"HAC t on the conditional leg (12m): {R['h_first'][12][7]:+.2f}")
print('\nThe timing device earns LESS per day invested than simply owning the asset.')

conditional leg (12m windows): +0.47%/y  95% CI [-2.05, +3.02]  share<0 36.6%  invested 18.7% of days
unconditional TLT - BIL      : +2.89%/y  95% CI [-3.64, +9.04]
HAC t on the conditional leg (12m): +0.35

The timing device earns LESS per day invested than simply owning the asset.


## Era split, cost sweep, borrow sweep, cross-checks

In [6]:
print(f"era split (2020-01-01), 12m: pre-2020 (N=2) {R['era_early']:+.2f}%  |  "
      f"2020+ (N=2) {R['era_late']:+.2f}%   <- complete sign flip")
print(f"cost sweep 12m: 0bps {R['cost0']:+.2f}%  5bps {R['cost5']:+.2f}%  "
      f"10bps {R['cost10']:+.2f}%  25bps {R['cost25']:+.2f}% (t={R['cost25_t']:+.2f})")
print(f"curve leg (long TLT / short SHY, 12m), borrow ASSUMPTION swept:")
print(f"   0 bp/y {R['ls_b0']:+.2f}% (t={R['ls_b0_t']:+.2f})   "
      f"50 bp/y {R['ls_b50']:+.2f}% (t={R['ls_b50_t']:+.2f})   "
      f"200 bp/y {R['ls_b200']:+.2f}% (t={R['ls_b200_t']:+.2f})")
print(f"IEF (belly) cross-check: 6m {R['ief_6m']:+.2f}% (t={R['ief_6m_t']:+.2f})  "
      f"12m {R['ief_12m']:+.2f}% (t={R['ief_12m_t']:+.2f})")
print(f"^IRX PROXY cash cross-check (12m): {R['irx_mean']:+.2f}% (t={R['irx_t']:+.2f})  "
      f"-> the cash leg is not what removes 2001; TLT's 2002 inception is")

era split (2020-01-01), 12m: pre-2020 (N=2) +19.43%  |  2020+ (N=2) -9.66%   <- complete sign flip
cost sweep 12m: 0bps +4.98%  5bps +4.88%  10bps +4.78%  25bps +4.48% (t=+0.50)
curve leg (long TLT / short SHY, 12m), borrow ASSUMPTION swept:
   0 bp/y +3.01% (t=+0.36)   50 bp/y +2.51% (t=+0.30)   200 bp/y +1.01% (t=+0.12)
IEF (belly) cross-check: 6m +2.98% (t=+1.06)  12m +4.26% (t=+1.17)
^IRX PROXY cash cross-check (12m): +5.09% (t=+0.56)  -> the cash leg is not what removes 2001; TLT's 2002 inception is


> 💡 **In plain words** — costs are irrelevant here (two trades a year), borrow only makes the curve version worse, and every cross-check reproduces the same shape: positive point estimate, no significance.

## Power calibration — what a five-event test can and cannot see

The live cell runs the identical harness on synthetic worlds with a **known** planted six-month effect of +9%, and on matched nulls. Pooled across worlds the estimator is unbiased and sharp; world-by-world, the five-event *t* is a coin flip. This is a property of the design, not of the Fed.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from first_cut import data, strategy as st

for tag, ss in [('planted (+9% true)', 1.0), ('null    (0% true) ', 0.0)]:
    ts, pooled = [], []
    for prices, events, _ in data.synthetic_panel(n_worlds=12, signal_strength=ss):
        tbl = st.event_table(prices['duration'], prices['cash'], events, 6, 5.0)
        ts.append(st.one_sample_t(tbl['excess_net_pct'].to_numpy()))
        pooled.extend(tbl['excess_net_pct'].tolist())
    ts, pooled = np.array(ts), np.array(pooled)
    print(f'{tag}: pooled N={len(pooled)} mean {pooled.mean():+5.2f}% '
          f't={st.one_sample_t(pooled):+5.2f} | per-world 5-event |t|>2 in '
          f'{(np.abs(ts) > 2).sum()}/12')
print('\nSYNTHETIC DATA (machinery + power check) — never a real-tape number.')

planted (+9% true): pooled N=60 mean +8.55% t=+6.93 | per-world 5-event |t|>2 in 5/12


null    (0% true) : pooled N=60 mean -0.83% t=-0.74 | per-world 5-event |t|>2 in 1/12

SYNTHETIC DATA (machinery + power check) — never a real-tape number.


## Verdict

- **Signal — None.** Headline 12-month mean **+4.88%** net excess-of-cash, *t* = +0.55 on N = 4, randomisation *p* = 0.365 against matched random start dates. The daily conditional leg's HAC *t* is +0.35 and its bootstrap CI [-2.05%, +3.02%] straddles zero. No horizon reaches *t* = 1.1. The all-cuts control (+5.50%, N = 18, nominal *t* = +1.78 — oversized, because those 18 twelve-month windows overlap onto only 3 macro episodes; its HAC daily *t* is +0.22) *beats* the first-cut leg on the mean, so the hand-picked label carries no information. Era split flips sign (+19.4% / -9.7%). The synthetic control confirms the harness is unbiased (pooled null -0.83%, *t* = -0.74) and correctly powered in aggregate (pooled planted +8.55%, *t* = +6.93) — while demonstrating that at N = 5 a true +9% effect is found only 5/12 of the time.
- **Tradability — Mirage.** +0.47%/y for the conditional strategy across the full sample (+2.62%/y counting only the days it is invested) versus +2.89%/y for holding the asset unconditionally — it loses on either reading; the curve expression is +2.51% per event at a 50 bp/y borrow (*t* = +0.30); and the entire positive mean is one 2019-2020 window. Nothing here is sizeable, and nothing here is separable from luck.
- **The methodological finding.** The interesting result is not that the trade failed but that it *could not have succeeded*: a macro event that fires four times in twenty years does not generate enough independent observations to clear any honest inference bar. Treat every "the Fed always..." claim built on a handful of cycles the same way.